In [19]:
# Install PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")

import torch.nn as nn

import tiktoken

PyTorch version: 2.9.1


## GPT Model

Implement the LLM model architecture used in OpenAI GPT.

The purpose of a *Normalization Layer* is to improve the numerical stability and speedup convergence by having a distribution with zero mean and unit variance, which avoids vanishing or exploding gradients. 
In GTP-2 it is applied before and after multi-head attention, before the final output layer.

In [20]:
# Set model configuration
GPT_CONFIG_124M = {
  "vocab_size": 50257,    # Vocabulary size
  "context_length": 1024, # Context length
  "emb_dim": 768,         # Embedding dimension
  "n_heads": 12,          # Number of attention heads
  "n_layers": 12,         # Number of layers
  "drop_rate": 0.1,       # Dropout rate
  "qkv_bias": False       # Query-Key-Value bias
}

In [3]:
# GPT model
class DummyGPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    # Create embedding layers
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    # Placeholder for TransformerBlock
    self.trf_blocks = nn.Sequential(
      *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
    )

    # Use a placeholder for LayerNorm
    self.final_norm = DummyLayerNorm(cfg["emb_dim"])
    self.out_head = nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )
  
  # Forward pass computation
  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)
    return logits

class DummyTransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()

  def forward(self, x):
    return x

class DummyLayerNorm(nn.Module):
  def __init__(self, normalized_shape, eps=1e-5):
    super().__init__()

  def forward(self, x):
    return x

In [9]:
# Use GPT tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Test tokenizing with two batches
batch = []
txt_1 = "Every effort moves you"
txt_2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt_1)))
batch.append(torch.tensor(tokenizer.encode(txt_2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [10]:
# Test dummy model
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


In [ ]:
# Test normalizing an activation layer
torch.manual_seed(123)
batch_example = torch.randn(2, 5)

# Create an activation layer
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print(f"Layer outputs: {out}")
print(f"Mean: {mean}")
print(f"Var: {var}")

# Normalize the layer outputs
out_norm = (out - mean) / torch.sqrt(var)
print(f"\nNormalized layer outputs: {out_norm}")
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print(f"Mean: {mean}")
print(f"Var: {var}")

tensor([[-0.1115,  0.1204, -0.3696, -0.2404, -1.1969],
        [ 0.2093, -0.9724, -0.7550,  0.3239, -0.1085]])
Layer outputs: tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)
Mean: tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
Var: tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)

Normalized layer outputs: tensor([[ 0.6159,  1.4126, -0.8719,  0.5872, -0.8719, -0.8719],
        [-0.0189,  0.1121, -1.0876,  1.5173,  0.5647, -1.0876]],
       grad_fn=<DivBackward0>)
Mean: tensor([[9.9341e-09],
        [1.9868e-08]], grad_fn=<MeanBackward1>)
Var: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [ ]:
# Normalization layer
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()

    # Small threshold constant
    self.eps = 1e-5

    # Scale and shift trainable layer
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.zeros(emb_dim))

  # Forward pass computation
  def forward(self, x):
    # Compute mean and variance of input tensor
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)

    # For normalization, add a small denominator constant to prevent division by zero
    norm_x = (x - mean) / torch.sqrt(var + self.eps)

    # Scale and shift
    return self.scale * norm_x + self.shift

In [15]:
# Test layer normalization
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)
print(f"Mean: {mean}")
print(f"Variance: {var}")

Mean: tensor([[-2.9802e-08],
        [ 0.0000e+00]], grad_fn=<MeanBackward1>)
Variance: tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


## GELU Activations

Implement feed forward network with Gaussian Error Linear Unit (GELU) activations. It is based on a Gaussian cumulative distribution function which uses nonlinearity weight inputs. It is a smoother version of ReLU(), resulting in better gradient flow during backpropagation.

By contrast, the closely-related ReLU function gates its inputs by their sign. This hard cutoff can result in dead neurons - where gradients become zero for negative inputs.

GELU = 0.5 * x * (1 + tanh[sqrt(2/pi) * (x + 0.044715 * x**3)])

A GELU activation layer is usually sandwiched between two linear layers. It is used by BERT and GPT-2/3/4.

A more modern drop-in version with lower loss is Swish-Gated Linear Units (SwiGELU), used by PaLM and LLaMA.

In [16]:
# GELU network layer
class GELU(nn.Module):
  def __init__(self):
    super().__init__()

  # Compute GELU function
  def forward(self, x):
    return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi))
      * (x + 0.044715 * torch.pow(x, 3))))

In [ ]:
# Feed forward network
class FeedForward(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    
    self.layers = nn.Sequential(
      # Expand the network (e.g. 768 -> 3072)
      nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),

      # Learn from the network using GELU
      GELU(),

      # Compress the network (e.g. 3072 -> 768)
      nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
    )
  
  def forward(self, x):
    return self.layers(x)

In [ ]:
# Test GELU network
ffn = FeedForward(GPT_CONFIG_124M)
x = torch.rand(2, 3, 768)
out = ffn(x)
print(out.shape)

torch.Size([2, 3, 768])


## Shortcut Connections

The purpose of a shortcut connection in a neural network is to add back the original input for when gradients are so small they do not convey any learned knowledge.

In [39]:
class ExampleDeepNeuralNetwork(nn.Module):
  def __init__(self, layer_sizes, use_shortcut):
    super().__init__()
    self.use_shortcut = use_shortcut
    self.layers = nn.ModuleList([
      nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
      nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
    ])

  def forward(self, x):
    for layer in self.layers:
      # Compute the output of the current layer
      layer_output = layer(x)

      # Check if shortcut can be applied
      if self.use_shortcut and x.shape == layer_output.shape:
        x = x + layer_output
      else:
        x = layer_output
    return x


def print_gradients(model, x):
  # Forward pass
  output = model(x)
  target = torch.tensor([[0.]])

  # Calculate loss based on how close the target and output are
  loss = nn.MSELoss()
  loss = loss(output, target)
  
  # Backward pass to calculate the gradients
  loss.backward()

  for name, param in model.named_parameters():
    if 'weight' in name:
      # Print the mean absolute gradient of the weights
      print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [42]:
# Test network without shortcuts
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=False)
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173584925942123
layers.1.0.weight has gradient mean of 0.00012011159560643137
layers.2.0.weight has gradient mean of 0.0007152040489017963
layers.3.0.weight has gradient mean of 0.0013988736318424344
layers.4.0.weight has gradient mean of 0.005049645435065031


In [ ]:
# Test network with shortcuts (to show gradients become larger)
layer_sizes = [3, 3, 3, 3, 3, 1]

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(layer_sizes, use_shortcut=True)
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.22169791162014008
layers.1.0.weight has gradient mean of 0.20694105327129364
layers.2.0.weight has gradient mean of 0.32896995544433594
layers.3.0.weight has gradient mean of 0.2665732204914093
layers.4.0.weight has gradient mean of 1.3258540630340576


## Transformer Block Connections

A transformer block combines a causal multi-head attention module with linear layers.

In [ ]:
# Previously defined
from production import MultiHeadAttention

class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    # Declare network layers
    self.att = MultiHeadAttention(
      d_in=cfg["emb_dim"],
      d_out=cfg["emb_dim"],
      context_length=cfg["context_length"],
      num_heads=cfg["n_heads"], 
      dropout=cfg["drop_rate"],
      qkv_bias=cfg["qkv_bias"])
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    # Shortcut connection for attention block
    shortcut = x
    x = self.norm1(x)
    x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
    x = self.drop_shortcut(x)
    x = x + shortcut  # Add the original input back

    # Shortcut connection for feed forward block
    shortcut = x
    x = self.norm2(x)
    x = self.ff(x)
    x = self.drop_shortcut(x)
    x = x + shortcut  # Add the original input back

    return x

In [44]:
# Test transformer block
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


## GPT Model Architecture

Implementation of the entire GPT architecture.

In [51]:
class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()

    # Declare input and output embedding layers
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

    # Declare dropout layer
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    # Declare transformer block layers
    self.xform_blocks = nn.Sequential(
      *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

    # Declare final normalization layer
    self.final_norm = LayerNorm(cfg["emb_dim"])

    # Declare linear layer
    self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape

    # Map vocabulary space to embedding space
    tok_embeds = self.tok_emb(in_idx)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))

    # Combine token and positional embeddings
    # Shape [batch_size, num_tokens, emb_size]
    x = tok_embeds + pos_embeds

    # Process dropout
    x = self.drop_emb(x)

    # Process transformer blocks
    x = self.xform_blocks(x)

    # Process final normalization
    x = self.final_norm(x)

    # Map embedding space to vocabulary space
    logits = self.out_head(x)
    return logits

In [52]:
# Test GPT model on 2x4 batch sample
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print(out.shape)
print(out)

torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


In [50]:
# Print number parameters in model
total_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {total_params:,}")

Number of parameters: 163,009,536


In [ ]:
# Generate text one token at a time
def generate_text_simple(model, idx, max_new_tokens, context_size):
  # Truncate current context if it exceeds the supported context size
  for _ in range(max_new_tokens):
    # Get token context
    idx_cond = idx[:, -context_size:]

    # Get the predictions from the tokens.
    # Disable gradient computation to suppress the model training.
    with torch.no_grad():
      logits = model(idx_cond)

    # Focus only on the last time step
    logits = logits[:, -1, :]

    # Apply softmax to get probabilities
    probas = torch.softmax(logits, dim=-1)

    # Get the idx of the vocab entry with the highest probability value
    idx_next = torch.argmax(probas, dim=-1, keepdim=True)

    # Append sampled index to the running sequence
    idx = torch.cat((idx, idx_next), dim=1)

  return idx

In [ ]:
# Prepare for text generation test
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])


In [ ]:
# Test text generation

# Set evaluation mode to disable dropout during inference
model.eval()

# Generate text
out = generate_text_simple(
  model=model,
  idx=encoded_tensor, 
  max_new_tokens=6, 
  context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716, 27018, 24086, 47843, 30961, 42348,  7267]])
Output length: 10


In [ ]:
# Print the untrained model output
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am Featureiman Byeswickattribute argue
